# FingerBall Stage 2J-1 — Multi-model 2D / 3D Hand Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_Stage2J1_MultiModel_2D3D_Comparison.ipynb)

### Goal
**Stage 2Hの目視確認済みXYを変更せず**、MediaPipe Tasks Hand LandmarkerとWiLoR-miniを同じ元動画フレームに推論させ、2D画素座標、推定3D、手の左右、モデル間の誤差を統一表で比較します。

- 元動画 1206×648 / frame 185–198、特に188–193（190はリリース目視参照、未自動検出）。
- 2D座標はすべて**元動画のピクセル座標**へ変換。モデルが異なる3D座標は**生値を別の座標系として記録**し、直接平均しません。
- 入力モデルの選択について**手動ROIを使用する条件（oracle-assisted）**と、モデル独立検出の条件を区別します。ROI assistedの2D誤差は**独立した検出性能の評価ではありません**。
- 遮蔽点は評価から除外。手動点、目視確認済み引継ぎ候補、AI点の出自を保持。
- 失敗した検出は欠損として出力し、成功したように補間しません。
- WiLoR-miniは内部で**MANO_RIGHT.pklをHugging Faceから自動取得**する実装です。MANOのモデルライセンスを必ず確認し、許諾を得るまで`ENABLE_WILOR=False`で実行してください。<https://mano.is.tue.mpg.de/>

参考：<https://github.com/warmshao/WiLoR-mini>、<https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python>

**これは比較研究用Notebookで、SOMAフィッティングは次のStage 2J-2です。**

## 0. Colabセットアップ：MediaPipe単独は常に実行可能

WiLoR-miniは旧Python 3.10想定で、最新Colabでは依存ライブラリとMANOライセンスの問題があり得ます。比較表はWiLoRが動かない場合も生成します。

In [ ]:
%pip -q install "mediapipe>=0.10.20" "opencv-python-headless>=4.8" "pandas>=2" "numpy>=1.24" "matplotlib>=3.7" "plotly>=5.18"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json,time,traceback,urllib.request,subprocess,sys
import numpy as np,pandas as pd,cv2,matplotlib.pyplot as plt
import mediapipe as mp
from IPython.display import display
print('Python',sys.version.split()[0],'MediaPipe',mp.__version__)

## 1. 設定・元データの読み込み

In [ ]:
PROJECT=Path('/content/drive/MyDrive/母指回内トラッキング')
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
VIDEO_PATH=PROJECT/(VIDEO_STEM+'.mp4')  # 加工済みzoom動画は使わない
STAGE2H_CSV=PROJECT/'FingerBall_v2_stage2h_results'/VIDEO_STEM/'hand_21_landmarks_2d_corrected.csv'
OUTPUT_DIR=PROJECT/'FingerBall_v2_stage2j1_results'/VIDEO_STEM
FRAME_START,FRAME_END=185,198
RELEASE_REFERENCE=190
THROWING_HAND='Left'
USE_MANUAL_ROI_FOR_MEDIAPIPE=True
ENABLE_WILOR=False   # MANOのライセンス・配布物の利用許諾を確認した場合のみ True
I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS=False
WILOR_RUN_DETECTOR_BASELINE=True
WILOR_RUN_MANUAL_BBOX_ASSIST=True
ROI_MARGIN=1.8
FRAME_DISPLAY=(188,189,190,191,192,193)
if THROWING_HAND not in ('Left','Right'):raise ValueError('Throwing hand must be Left/Right')
if ENABLE_WILOR and not I_HAVE_RIGHTS_TO_USE_WILOR_MANO_ASSETS:
    raise PermissionError('WiLoR-miniはMANOモデルファイルを自動取得します。MANOの利用許諾を確認してからフラグを設定してください')
for file in (VIDEO_PATH,STAGE2H_CSV):
    if not file.is_file():raise FileNotFoundError(file)
if 'zoom' in VIDEO_PATH.stem.lower():raise ValueError('元動画を指定してください')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
source_hash=hashlib.sha256(STAGE2H_CSV.read_bytes()).hexdigest()
ref=pd.read_csv(STAGE2H_CSV)
req={'frame','landmark_id','x_px','y_px','annotation_status'}
if diff:=req-set(ref.columns):raise ValueError(f'Missing Stage2H columns: {diff}')
if ref.duplicated(['frame','landmark_id']).any():raise ValueError('Duplicate rows')
frames=list(range(FRAME_START,FRAME_END+1))
ref=ref[ref.frame.isin(frames)].copy()
assert len(ref)==21*len(frames)
assert all(ref.groupby('frame').landmark_id.nunique().eq(21))
cap=cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():raise RuntimeError('Video failed to open')
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
assert FRAME_END<int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
assert W>0 and H>0
def frame_rgb(f):
    cap.set(cv2.CAP_PROP_POS_FRAMES,int(f))
    ok,bgr=cap.read()
    if not ok:raise RuntimeError(f'Could not read frame {f}')
    return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
images={f:frame_rgb(f) for f in frames}
cap.release()
reference={};status={}
for f in frames:
    a=ref[ref.frame==f].set_index('landmark_id').reindex(range(21))
    reference[f]=a[['x_px','y_px']].to_numpy(float)
    status[f]=a.annotation_status.astype(str).to_list()
print('Video',W,H,'FPS',FPS,'Frozen SHA256',source_hash)
print('Annotations',ref.annotation_status.value_counts().to_dict())

## 2. ROIと共通フォーマット

手動点に基づくROIは研究上**手動補助条件**です。独立した自動検出性能の評価と区別します。

In [ ]:
EDGE_SETS={'Thumb':[0,1,2,3,4],'Index':[0,5,6,7,8],
           'Middle':[0,9,10,11,12],'Ring':[0,13,14,15,16],
           'Little':[0,17,18,19,20]}
COLORS={'Thumb':'#d97706','Index':'#0891b2','Middle':'#be185d',
        'Ring':'#94a3b8','Little':'#64748b'}
NAME=['WRIST','THUMB_CMC','THUMB_MCP','THUMB_IP','THUMB_TIP',
      'INDEX_MCP','INDEX_PIP','INDEX_DIP','INDEX_TIP',
      'MIDDLE_MCP','MIDDLE_PIP','MIDDLE_DIP','MIDDLE_TIP',
      'RING_MCP','RING_PIP','RING_DIP','RING_TIP',
      'PINKY_MCP','PINKY_PIP','PINKY_DIP','PINKY_TIP']
def manual_box(f,margin=ROI_MARGIN):
    p=reference[f]
    p=p[np.isfinite(p).all(axis=1)]
    if len(p)<3:raise ValueError(f'Frame {f}: too few reference points for assisted ROI')
    lo=np.min(p,axis=0);hi=np.max(p,axis=0)
    center=(lo+hi)/2
    side=max(float(max(hi-lo))*margin,100.)
    x1=max(0,center[0]-side/2);y1=max(0,center[1]-side/2)
    x2=min(W,center[0]+side/2);y2=min(H,center[1]+side/2)
    if x2<=x1 or y2<=y1:raise ValueError('Invalid box')
    return np.array([x1,y1,x2,y2],float)
def image_crop(image,box,size=512):
    x1,y1,x2,y2=box
    side=max(x2-x1,y2-y1);cx=(x1+x2)/2;cy=(y1+y2)/2
    scale=size/side
    M=np.float32([[scale,0,size/2-cx*scale],[0,scale,size/2-cy*scale]])
    patch=cv2.warpAffine(image,M,(size,size),flags=cv2.INTER_LINEAR,
               borderMode=cv2.BORDER_CONSTANT,borderValue=(105,105,105))
    Minv=cv2.invertAffineTransform(M)
    return patch,Minv
def map_to_original(p,Minv):
    return np.concatenate([p,np.ones((len(p),1))],axis=1)@Minv.T
empty21=lambda:np.full((21,3),np.nan,float)
predictions={}
def store(f,method,xy=None,world=None,relative=None,handedness='',confidence=np.nan,
          detected=False,reason='',assisted=False):
    if xy is None:xy=np.full((21,2),np.nan)
    if world is None:world=empty21()
    if relative is None:relative=np.full((21,),np.nan)
    xy=np.asarray(xy,dtype=float).reshape(21,2)
    world=np.asarray(world,dtype=float).reshape(21,3)
    relative=np.asarray(relative,dtype=float).reshape(21)
    predictions[(int(f),method)]=dict(xy=xy,world=world,relative=relative,
         handedness=str(handedness),confidence=float(confidence),
         detected=bool(detected),reason=str(reason),assisted=bool(assisted))
def choose_hand(candidates,refxy,requested='Left'):
    # candidate dict: xy + hand label, score
    if not candidates:return None
    ranked=[]
    for c in candidates:
        good=np.isfinite(refxy).all(axis=1)&np.isfinite(c['xy']).all(axis=1)
        dist=float(np.median(np.linalg.norm(refxy[good]-c['xy'][good],axis=1))) if np.any(good) else 1e7
        handedness_penalty=0 if c.get('hand','')==requested else 200
        ranked.append((dist+handedness_penalty,c))
    return min(ranked,key=lambda v:v[0])[1]